<a href="https://colab.research.google.com/github/jazminfuentesb/ejercicios_ingenieria_en_datos_e_ia/blob/main/Ejercicios_23_03_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Resolución mediante el método simplex

In [2]:
#Bibliotecas
import numpy as np

In [18]:
# ============================================================
# FUNCIONES BASE
# ============================================================

def pivotear(tableau, fila, col):
    pivote = tableau[fila, col]
    tableau[fila, :] = tableau[fila, :] / pivote

    for i in range(len(tableau)):
        if i != fila:
            tableau[i, :] -= tableau[i, col] * tableau[fila, :]
    return tableau


def simplex(tableau):
    while True:
        # Selección de columna pivote
        col = np.argmin(tableau[-1, :-1])

        if tableau[-1, col] >= 0:
            break  # óptimo

        ratios = []
        for i in range(len(tableau)-1):
            if tableau[i, col] > 0:
                ratios.append(tableau[i, -1] / tableau[i, col])
            else:
                ratios.append(np.inf)

        fila = np.argmin(ratios)

        if ratios[fila] == np.inf:
            raise ValueError("Problema no acotado")

        tableau = pivotear(tableau, fila, col)

    return tableau




## Ejercicio dual 1

In [4]:
# ============================================
# EJERCICIO 1: Maximización de Beneficios
# ============================================
# Contexto:
# Una empresa produce dos productos, A y B.
# Cada unidad de A genera $2 y cada unidad de B genera $3.
#
# Restricciones:
# 1. Máquina: máximo 480 minutos
#    - A requiere 1 min/unidad
#    - B requiere 2 min/unidad
#
# 2. Trabajo manual: máximo 960 minutos
#    - A requiere 4 min/unidad
#    - B no requiere trabajo manual
#
# 3. Empaque: máximo 960 minutos
#    - A no requiere empaque
#    - B requiere 4 min/unidad
#
# Objetivo:
# Maximizar Z = 2x1 + 3x2
#
# Restricciones matemáticas (normalizadas):
# x1 + 2x2 <= 8
# 4x1 <= 16
# 4x2 <= 16
# x1, x2 >= 0
# ============================================================
# FORMULACIÓN DUAL
# ============================================================
# Min Z = 480y1 + 960y2 + 960y3
#
# Restricciones:
# y1 + 4y2 >= 1
# 2y1 + 4y3 >= 1
#
# Convertimos a forma <= multiplicando por -1

In [17]:
# Función objetivo (incluye variables de holgura)
c = np.array([480, 960, 960, 0, 0], dtype=float)

# Restricciones convertidas a <= + variables de holgura
A = np.array([
    [-1, -4, 0, 1, 0],
    [-2, 0, -4, 0, 1]
], dtype=float)

b = np.array([-1, -1], dtype=float)

# Tableau inicial
tableau = np.vstack([
    np.hstack([A, b.reshape(-1, 1)]),
    np.hstack([-c, np.array([0.0])])
])


# ============================================================
# EJECUCIÓN
# ============================================================

tableau_final = metodo_simplex(tableau)


# ============================================================
# RESULTADOS
# ============================================================

solucion = tableau_final[:-1, -1]
valor_optimo = -tableau_final[-1, -1]

print("============================================")
print("RESULTADOS FINALES")
print("============================================")

# Solo mostramos y1, y2, y3 (las primeras 3 variables)
print("y1, y2, y3 =", solucion[:3])
print("Costo mínimo =", valor_optimo)

Solución óptima encontrada.

RESULTADOS FINALES
y1, y2, y3 = [-1. -1.]
Costo mínimo = -0.0


Durante el desarrollo de la actividad, se implementó el método simplex utilizando el código proporcionado por el docente, con el objetivo de resolver los problemas duales planteados. Sin embargo, al ejecutar el algoritmo se identificaron diversas limitaciones que impidieron obtener soluciones válidas.

En particular, el código presenta dificultades para manejar restricciones del tipo ≥, ya que no incorpora de manera completa los elementos necesarios para el método simplex dual, como el uso adecuado de variables artificiales, la construcción de una base inicial factible y la aplicación formal del método de dos fases. Como consecuencia, el algoritmo genera errores numéricos, selección incorrecta de pivotes o soluciones que no cumplen con las condiciones de factibilidad (por ejemplo, valores negativos en variables que deben ser no negativas).

Aunque se realizaron ajustes al código original, como el uso de tipos de datos flotantes y mejoras en el criterio de selección de pivotes, estas modificaciones no fueron suficientes para garantizar la convergencia del método hacia una solución óptima válida en todos los casos.

Debido a estas limitaciones, se optó por utilizar la función linprog de la biblioteca SciPy como herramienta complementaria para la resolución de los problemas. Esta función implementa algoritmos avanzados de optimización lineal, incluyendo variantes del método simplex, lo que permite obtener soluciones numéricamente estables y consistentes con la formulación teórica del problema.

Finalmente, es importante destacar que el uso de esta herramienta no sustituye la comprensión del método simplex, sino que permite validar los resultados obtenidos y evidenciar las limitaciones de una implementación incompleta del algoritmo.

In [22]:
# Importamos la función linprog de SciPy,
# la cual permite resolver problemas de programación lineal
from scipy.optimize import linprog


# ============================================================
# FUNCIÓN OBJETIVO
# ============================================================
# Min Z = 480y1 + 960y2 + 960y3
# Representa el costo asociado a cada recurso (variables duales)

c = [480, 960, 960]


# ============================================================
# RESTRICCIONES
# ============================================================
# El problema original tiene restricciones tipo ≥:
# y1 + 4y2 >= 1
# 2y1 + 4y3 >= 1
#
# linprog trabaja con restricciones tipo ≤,
# por lo que multiplicamos por -1:
#
# -y1 - 4y2 <= -1
# -2y1 - 4y3 <= -1

A = [
    [-1, -4, 0],   # -y1 - 4y2 <= -1
    [-2, 0, -4]    # -2y1 - 4y3 <= -1
]

# Lado derecho de las desigualdades
b = [-1, -1]


# ============================================================
# RESTRICCIONES DE NO NEGATIVIDAD
# ============================================================
# y1, y2, y3 >= 0
# Se representan como límites (lower bound = 0, sin límite superior)

bounds = [
    (0, None),  # y1 >= 0
    (0, None),  # y2 >= 0
    (0, None)   # y3 >= 0
]


# ============================================================
# RESOLUCIÓN DEL PROBLEMA
# ============================================================
# Se utiliza el método 'highs', el cual implementa algoritmos
# avanzados (incluyendo variantes del método simplex dual)

res = linprog(
    c,          # coeficientes de la función objetivo
    A_ub=A,     # matriz de restricciones (lado izquierdo)
    b_ub=b,     # vector del lado derecho
    bounds=bounds,
    method='highs'
)


# ============================================================
# RESULTADOS
# ============================================================
# res.x → valores óptimos de las variables (y1, y2, y3)
# res.fun → valor mínimo de la función objetivo

print("y1, y2, y3 =", res.x)
print("Costo mínimo =", res.fun)

y1, y2, y3 = [0.5   0.125 0.   ]
Costo mínimo = 360.0


## Ejercicio dual 2

In [25]:
# ============================================================
# EJERCICIO 2: Asignación de Recursos
# ============================================================
# Contexto:
# Se tienen dos recursos (R1 y R2) que deben asignarse a dos proyectos (P1 y P2).
# El objetivo es minimizar el costo asociado al cumplimiento de las restricciones,
# utilizando variables duales.


# ============================================================
# FUNCIÓN OBJETIVO
# ============================================================
# Min Z = 8y1 + 16y2
# Donde:
# y1 = costo asociado al recurso 1
# y2 = costo asociado al recurso 2

c = [8, 16]


# ============================================================
# RESTRICCIONES
# ============================================================
# Del problema:
# y1 + y2 >= 2   (proyecto P1)
# y1 + y2 >= 3   (proyecto P2)
#
# linprog solo trabaja con ≤, entonces:
# multiplicamos por -1:
#
# -y1 - y2 <= -2
# -y1 - y2 <= -3


In [26]:
# ============================================================
# FUNCIÓN OBJETIVO
# ============================================================
# Min Z = 8y1 + 16y2
# Donde:
# y1 = costo asociado al recurso 1
# y2 = costo asociado al recurso 2

c = [8, 16]


# ============================================================
# RESTRICCIONES
# ============================================================
# Del problema:
# y1 + y2 >= 2   (proyecto P1)
# y1 + y2 >= 3   (proyecto P2)
#
# linprog solo trabaja con ≤, entonces:
# multiplicamos por -1:
#
# -y1 - y2 <= -2
# -y1 - y2 <= -3

A = [
    [-1, -1],   # restricción 1
    [-1, -1]    # restricción 2
]

# Lado derecho de las desigualdades
b = [-2, -3]


# ============================================================
# RESTRICCIONES DE NO NEGATIVIDAD
# ============================================================
# y1, y2 >= 0

bounds = [
    (0, None),  # y1 >= 0
    (0, None)   # y2 >= 0
]


# ============================================================
# RESOLUCIÓN DEL PROBLEMA
# ============================================================
# Se usa el método 'highs', que implementa algoritmos modernos
# de optimización lineal (incluyendo simplex dual)

res = linprog(
    c,          # coeficientes de la función objetivo
    A_ub=A,     # matriz de restricciones
    b_ub=b,     # vector lado derecho
    bounds=bounds,
    method='highs'
)


# ============================================================
# RESULTADOS
# ============================================================
# res.x → valores óptimos de las variables
# res.fun → valor mínimo de la función objetivo

print("y1, y2 =", res.x)
print("Costo mínimo =", res.fun)

y1, y2 = [3. 0.]
Costo mínimo = 24.0


## Ejercicio dual 3

In [27]:
# ============================================================
# EJERCICIO 3: Dieta Óptima
# ============================================================
# Contexto:
# Se busca minimizar el costo de una dieta cumpliendo restricciones
# de proteínas, calorías y presupuesto, utilizando variables duales.


# ============================================================
# FUNCIÓN OBJETIVO
# ============================================================
# Min Z = 8y1 + 16y2 + 16y3
#
# Donde:
# y1 = costo asociado a la restricción de proteínas
# y2 = costo asociado a calorías
# y3 = costo asociado al presupuesto

c = [8, 16, 16]


# ============================================================
# RESTRICCIONES
# ============================================================
# Del problema:
# y1 >= 1
# 2y1 + 4y3 >= 4

In [29]:
# Convertimos a formato ≤ (requerido por linprog):
#
# -y1 <= -1
# -2y1 - 4y3 <= -4

A = [
    [-1, 0, 0],    # -y1 <= -1
    [-2, 0, -4]    # -2y1 - 4y3 <= -4
]

# Lado derecho de las desigualdades
b = [-1, -4]


# ============================================================
# RESTRICCIONES DE NO NEGATIVIDAD
# ============================================================
# y1, y2, y3 >= 0

bounds = [
    (0, None),  # y1 >= 0
    (0, None),  # y2 >= 0
    (0, None)   # y3 >= 0
]


# ============================================================
# RESOLUCIÓN DEL PROBLEMA
# ============================================================
# Se utiliza el método 'highs', que incluye variantes modernas
# del método simplex (dual y primal) y otros algoritmos eficientes

res = linprog(
    c,          # coeficientes de la función objetivo
    A_ub=A,     # matriz de restricciones
    b_ub=b,     # vector lado derecho
    bounds=bounds,
    method='highs'
)


# ============================================================
# RESULTADOS
# ============================================================
# res.x → valores óptimos de las variables
# res.fun → valor mínimo de la función objetivo

print("y1, y2, y3 =", res.x)
print("Costo mínimo =", res.fun)

y1, y2, y3 = [2. 0. 0.]
Costo mínimo = 16.0
